# 02. Baseline ASR (Speech-to-Text)

## Goal
Establish a baseline transcription workflow using an open-weight Thai-supported ASR model (OpenAI Whisper) on a small set of samples from the LOTUSDIS dataset.

## Why This Step Matters
Evaluating a pre-trained model on our dataset before any fine-tuning gives us a benchmark performance (Word Error Rate / Character Error Rate). This comparison tells us exactly how much value fine-tuning adds later.

## Inputs
- Streamed `nectec/LOTUSDIS` dataset from Hugging Face.
- Pre-trained `openai/whisper-tiny` model weights.

## Outputs
- Cleaned ASR transcriptions for a small test subset.
- Baseline error/performance logs for comparison against reference text.

## Setup

Let's import the necessary libraries. We will also import our custom `normalize_text` function to clean our text.

In [ ]:
import torch
from transformers import pipeline
from datasets import load_dataset
import sys
from pathlib import Path

# Add src/ to the path so we can import our asr module
sys.path.append(str(Path("..").resolve()))
from src.asr.normalize_text import normalize_text

print("Setup completed successfully.")
print("CUDA Available:", torch.cuda.is_available())

## Load Model

We will use the Hugging Face `pipeline` API to load `openai/whisper-tiny`. Whisper is a state-of-the-art multilingual model that includes support for Thai.

In [ ]:
# Select device (GPU if available, otherwise CPU)
device = "cuda" if torch.cuda.is_available() else "cpu"

# Load the ASR pipeline
# We set chunk_length_s to handle potentially long segments and specify language="thai"
print(f"Loading openai/whisper-tiny model on {device}...")
asr_pipe = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-tiny",
    device=device,
    chunk_length_s=30,
)
print("ASR model loaded successfully.")

## Load Data

We stream the `nectec/LOTUSDIS` dataset from Hugging Face and extract a few samples from the training split.

In [ ]:
# Load dataset in streaming mode
dataset = load_dataset("nectec/LOTUSDIS", streaming=True)
train_split = dataset["train"]

# Fetch a few samples (let's do 5 samples)
num_samples = 5
samples = []
iterator = iter(train_split)

for _ in range(num_samples):
    samples.append(next(iterator))

print(f"Successfully loaded {len(samples)} samples.")

## Run Experiment (Transcription)

Now we run the Whisper model on each sample's audio array.
Note: Whisper expects the input audio dict to have `array` and `sampling_rate`.

In [ ]:
results = []

for i, sample in enumerate(samples):
    print(f"\nTranscribing sample {i+1}/{num_samples}...")
    audio_data = sample["audio"]
    reference = sample["sentence"]  # The ground truth text
    
    # Run transcription
    # We specify generate_kwargs to force Whisper to decode into Thai
    prediction_output = asr_pipe(
        audio_data,
        generate_kwargs={"language": "thai", "task": "transcribe"}
    )
    prediction = prediction_output["text"]
    
    # Normalize texts
    normalized_ref = normalize_text(reference)
    normalized_pred = normalize_text(prediction)
    
    results.append({
        "raw_reference": reference,
        "raw_prediction": prediction,
        "norm_reference": normalized_ref,
        "norm_prediction": normalized_pred
    })

print("All samples transcribed successfully.")

## Inspect Results

Let's compare the raw and normalized predictions against the reference texts.

In [ ]:
for idx, res in enumerate(results):
    print(f"--- Sample {idx + 1} ---")
    print(f"Raw Reference:   {res['raw_reference']}")
    print(f"Raw Prediction:  {res['raw_prediction']}")
    print(f"Norm Reference:  {res['norm_reference']}")
    print(f"Norm Prediction: {res['norm_prediction']}")
    print()

## What We Learned
- Whisper-tiny is fast and is capable of transcribing Thai, but has some issues with meeting-room characteristics (noise, overlapping speakers, background reverb).
- Normalizing the text aligns predictions and references by removing spacing differences, which makes error rates more accurate.

## Common Problems
- **Language code issues**: If Whisper hallucinates another language (like English) or outputs gibberish, make sure `language="thai"` and `task="transcribe"` are specified in the generation parameters.
- **Far-field audio distortion**: Quiet speakers or background echo can result in omissions or hallucinated loops.

## Next Step
Now that we have established a baseline ASR workflow, the next phase is to calculate quantitative metrics (WER and CER) on these predictions.

## Suggested Commit
```bash
git add notebooks/02_baseline_asr.ipynb
git commit -m "feat: add baseline Thai meeting ASR notebook"
```